# Fake Profile Detector — Training Notebook (multi-platform)

**Run on Google Colab — takes ~5 minutes end to end.**

## Datasets used

| Platform | Source | Type |
|----------|--------|------|
| **Instagram** | `free4ever1/instagram-fake-spammer-genuine-accounts` (Kaggle) | Real labeled data |
| **Twitter / X** | tries several known Kaggle slugs in order; auto-falls back to a high-quality **synthetic generator** modelled on published bot research (Cresci-2017, TwiBot-20 distributions) | Real if available, else synthetic |
| **Facebook** | Synthetic generator based on known fake-profile feature distributions (Meta restricts public FB datasets — synthesizing from research is the standard approach) | Synthetic |

> Why synthetic for TW/FB? Public Twitter/Facebook profile datasets on Kaggle are unstable and slugs often disappear. Synthetic data generated from realistic feature distributions trains a perfectly usable platform-aware model. The Instagram portion provides the real ML signal; the synthetic portions add platform diversity so the model doesn't overfit to IG-only signals.

## Steps
1. **Cell 1** — install deps
2. **Cell 2** — upload your `kaggle.json` (one-time; see instructions in cell)
3. **Cell 3** — download real Instagram + try real Twitter
4. **Cell 4** — generate synthetic Twitter/Facebook backup data
5. **Cell 5** — harmonize all into one schema
6. **Cell 6** — train LightGBM
7. **Cell 7** — save & download `model.pkl` + `feature_schema.json`

**Click `Runtime → Run all`** and you're done.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q kaggle lightgbm scikit-learn pandas numpy joblib

## Cell 2 — Upload your kaggle.json (one-time)

Get it from kaggle.com → click your avatar → **Settings** → **Create New Token** → downloads `kaggle.json`.

In [ ]:
import os, json
from google.colab import files

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Please upload your kaggle.json file (kaggle.com → Settings → Create New Token)')
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(list(uploaded.values())[0])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('kaggle.json saved.')
else:
    print('kaggle.json already present.')

## Cell 3 — Download real datasets (with multi-slug fallbacks)

Tries several known Kaggle slugs for each platform; uses whichever works first.

In [ ]:
import os, subprocess, glob, shutil, json
import pandas as pd

def try_kaggle(candidates, dest_folder):
    """Try a list of Kaggle dataset slugs; return list of slugs that worked."""
    os.makedirs(dest_folder, exist_ok=True)
    succeeded = []
    for slug in candidates:
        sub = os.path.join(dest_folder, slug.split('/')[-1])
        os.makedirs(sub, exist_ok=True)
        if any(f.endswith('.csv') for f in os.listdir(sub)):
            print(f'  (already have) {slug}')
            succeeded.append(slug)
            continue
        print(f'  trying {slug} ...', end=' ', flush=True)
        r = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', slug, '-p', sub, '--unzip'],
            capture_output=True, text=True
        )
        if r.returncode == 0 and any(f.endswith('.csv') for f in os.listdir(sub)):
            print('OK')
            succeeded.append(slug)
        else:
            err = r.stderr.strip().splitlines()[-1] if r.stderr else 'failed'
            print(f'FAIL ({err[:80]})')
            try:
                if not os.listdir(sub): os.rmdir(sub)
            except: pass
    return succeeded


def try_github(repo: str, dest_folder: str) -> str | None:
    """Clone repo and copy CSV + JSON files into dest_folder/<repo-name>/.
    Returns the destination subfolder path if anything was copied, else None."""
    name = repo.split('/')[-1]
    clone_dir = f'/tmp/{name}'
    os.makedirs(dest_folder, exist_ok=True)
    if not os.path.exists(clone_dir):
        print(f'  cloning {repo} ...', end=' ', flush=True)
        r = subprocess.run(
            ['git', 'clone', '--depth', '1', f'https://github.com/{repo}.git', clone_dir],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print(f'FAIL ({r.stderr.strip().splitlines()[-1][:80] if r.stderr else "clone failed"})')
            return None
        print('OK')
    sub = os.path.join(dest_folder, name)
    os.makedirs(sub, exist_ok=True)
    n_csv = n_json = 0
    for ext in ('csv', 'json'):
        for path in glob.glob(os.path.join(clone_dir, '**', f'*.{ext}'), recursive=True):
            size_mb = os.path.getsize(path) / 1e6
            # Skip tiny non-data JSONs (package.json, manifest.json, etc.)
            if ext == 'json' and size_mb < 0.05:
                continue
            try:
                shutil.copy(path, os.path.join(sub, os.path.basename(path)))
                if ext == 'csv': n_csv += 1
                else:            n_json += 1
            except Exception as e:
                print(f'   copy fail: {e}')
    print(f'  -> copied {n_csv} CSV + {n_json} JSON from {repo}')
    return sub if (n_csv + n_json) > 0 else None


def convert_twibot20(folder: str) -> bool:
    """TwiBot-20 stores user metadata in node.json and bot/human labels in label.csv.
    Join them and write a single twibot20_users.csv that the harmonizer can consume.

    node.json schema (per user):
      {"ID": "...", "username": "...", "name": "...", "description": "...",
       "followers_count": N, "friends_count": N, "statuses_count": N,
       "default_profile_image": true/false, "verified": true/false, ...}
    label.csv: id,label  where label is 'bot' or 'human'.
    """
    node_path = label_path = None
    for p in glob.glob(os.path.join(folder, '*')):
        bn = os.path.basename(p).lower()
        if bn == 'node.json':   node_path = p
        if bn == 'label.csv':   label_path = p
    if not (node_path and label_path):
        return False
    try:
        print('  converting TwiBot-20 node.json + label.csv -> CSV ...', end=' ', flush=True)
        with open(node_path, 'r', encoding='utf-8') as f:
            nodes = json.load(f)
        # nodes can be a list of dicts or a dict keyed by id
        if isinstance(nodes, dict):
            nodes = list(nodes.values())
        labels = pd.read_csv(label_path)
        # standardize columns
        labels.columns = [c.lower() for c in labels.columns]
        id_col = 'id' if 'id' in labels.columns else labels.columns[0]
        lab_col = 'label' if 'label' in labels.columns else labels.columns[1]
        labels = labels.rename(columns={id_col: 'id', lab_col: 'label'})

        rows = []
        for n in nodes:
            uid = str(n.get('ID') or n.get('id') or n.get('id_str') or '')
            if not uid: continue
            rows.append({
                'id': uid,
                'screen_name':           n.get('username') or n.get('screen_name') or '',
                'name':                  n.get('name') or '',
                'description':           n.get('description') or '',
                'followers_count':       int(n.get('followers_count') or n.get('public_metrics', {}).get('followers_count', 0) or 0),
                'friends_count':         int(n.get('friends_count')   or n.get('following_count') or n.get('public_metrics', {}).get('following_count', 0) or 0),
                'statuses_count':        int(n.get('statuses_count')  or n.get('public_metrics', {}).get('tweet_count', 0) or 0),
                'default_profile_image': int(bool(n.get('default_profile_image'))),
            })
        nodes_df = pd.DataFrame(rows)
        nodes_df['id'] = nodes_df['id'].astype(str)
        labels['id'] = labels['id'].astype(str)
        merged = nodes_df.merge(labels[['id', 'label']], on='id', how='inner')
        merged['account_type'] = merged['label']  # so harmonize_tw picks up 'bot'/'human'
        out_path = os.path.join(folder, 'twibot20_users.csv')
        merged.to_csv(out_path, index=False)
        print(f'OK ({len(merged)} rows -> {out_path})')
        return True
    except Exception as e:
        print(f'FAIL ({type(e).__name__}: {str(e)[:80]})')
        return False


# =============================================================================
# INSTAGRAM
# =============================================================================
print('=== INSTAGRAM (Kaggle) ===')
ig_slugs = try_kaggle([
    'free4ever1/instagram-fake-spammer-genuine-accounts',
    'rezaunderfit/instagram-fake-and-real-accounts-dataset',
    'krpurba/fakeauthentic-user-instagram',
    'bhanuprakashm/fake-profile-detection-using-machinelearning',
    'tahirasultana1/instagram-fake-and-real-accounts',
    'gusergiusergaming/instagram-fake-account-dataset',
    'sumeetgodara/instagramfakeaccount-detection-using-lstm',
    'harshilshah1011/fake-instagram-accounts-dataset',
    'karanrana/instagram-fake-spammer-genuine-accounts',
], 'data/ig')

print('\n=== INSTAGRAM (GitHub) ===')
try_github('Bharti-Goyal/Instagram-fake-account-dataset', 'data/ig')

# =============================================================================
# TWITTER / X
# =============================================================================
print('\n=== TWITTER / X (Kaggle) ===')
tw_slugs = try_kaggle([
    'goyaladi/twitter-bot-detection-dataset',
    'charvijain27/detecting-twitter-bot-data',
    'davidmartingarcia/twitter-bots-accounts',
    'kpriyanshu256/twitter-bot-data',
    'thedevastator/twitter-bot-and-genuine-accounts',
    'lcsouzamenezes/social-media-bot-detection-dataset',
    'krrai77/twitter-bot-detection-data',
], 'data/tw')

print('\n=== TWITTER / X (GitHub TwiBot-20) ===')
twibot_dir = try_github('BunsenFeng/TwiBot-20', 'data/tw')
if twibot_dir:
    convert_twibot20(twibot_dir)

# =============================================================================
# FACEBOOK
# =============================================================================
print('\n=== FACEBOOK (Kaggle) ===')
fb_slugs = try_kaggle([
    'khajahussainsk/facebook-spam-dataset',
    'bitandatom/social-network-fake-account-dataset',
    'rafsunahmad/fake-and-real-facebook-accounts',
    'kaustubhdhote/facebook-fake-account-dataset',
    'devvret/facebook-fake-account-dataset',
    'rezaghari/fake-profiles',
    'sankalpsrivastava01/fake-account-detection',
], 'data/fb')

print('\n=== Loaded ===')
def count_csvs(folder):
    return len(glob.glob(os.path.join(folder, '**', '*.csv'), recursive=True))
for p in ['ig', 'tw', 'fb']:
    n = count_csvs(f'data/{p}')
    subs = len(os.listdir(f'data/{p}')) if os.path.exists(f'data/{p}') else 0
    print(f'{p.upper()}: {n} CSV file(s) across {subs} subfolders')

## Cell 3b — (Optional) Manually upload extra CSVs

If you have any extra Kaggle CSVs you downloaded by hand, drop them into the right platform folder by running the cell below. Skip this cell if you don't need it.

The cell will let you pick which platform each upload belongs to (ig / tw / fb).

In [ ]:
# OPTIONAL — skip if you don't have extra CSVs.
# Change PLATFORM to 'ig', 'tw', or 'fb' depending on which platform the uploaded CSV(s)
# belong to. Then run the cell — Colab will prompt you to pick file(s) from your machine.

PLATFORM = 'fb'   # <-- change to 'ig' / 'tw' / 'fb' as needed

import os
from google.colab import files
dest = f'data/{PLATFORM}/manual_upload'
os.makedirs(dest, exist_ok=True)
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(os.path.join(dest, fname), 'wb') as f:
        f.write(content)
    print(f'Saved -> {os.path.join(dest, fname)}')

## Cell 4 — Synthetic data generators (used as fallback for any missing platform)

Distributions are based on published research on bot/fake account characteristics:
- **Bots/fakes:** disproportionately high `following`, low `followers`, often no profile pic, generic/missing bio, many digits in username, very few or zero posts.
- **Genuine:** balanced ratios, profile pic present, populated bio, posts > 0.

These distributions are intentionally noisy so the model learns realistic decision boundaries rather than memorizing rules.

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(42)

# Synthetic data is intentionally SMALL — used only to give the model platform diversity
# when real per-platform data is missing. Real Kaggle data dominates training.
#
# Distributions are deliberately WIDE so they overlap with the real-user space:
# - real users span tiny (10 followers, brand new) to huge (millions). We sample across.
# - fake users skew bot-like (low followers, high following, low posts) but with overlap.
# Reference: Cresci et al. 2017 (MIB), Khaled et al. 2018, Sansonetti 2020.

WORDS = ['john','sarah','mike','emma','david','liam','noah','olivia','sophia','ava','ethan',
         'tech','dev','code','art','photo','music','travel','food','fit','game','life','daily',
         'cool','best','real','pro','x','luna','star','sky','sun','moon','blue','red','green',
         'official','the','mr','ms','dr','your','my','our','user','account','profile','fan',
         'love','dream','hope','vibes','queen','king','smith','jones','brown','wilson','lee']


def fake_username(rng):
    style = rng.integers(0, 4)
    if   style == 0: return ''.join(str(rng.integers(0, 10)) for _ in range(rng.integers(8, 12)))
    elif style == 1: return ''.join(rng.choice(list('bcdfghjklmnpqrstvwxz'), size=rng.integers(7, 12)))
    elif style == 2: return rng.choice(WORDS) + ''.join(str(rng.integers(0,10)) for _ in range(rng.integers(7, 11)))
    else:            return 'user' + ''.join(str(rng.integers(0,10)) for _ in range(rng.integers(7, 11)))


def real_username(rng, platform):
    """Realistic genuine usernames. ~45% of real Twitter users have auto-suggested
    handles with digit suffixes (like 'saqibme03871376') — keep this so the model
    doesn't treat that pattern alone as a fake signal."""
    if platform == 'twitter' and rng.random() < 0.45:
        return rng.choice(WORDS) + ''.join(str(rng.integers(0,10)) for _ in range(rng.integers(6, 10)))
    style = rng.integers(0, 5)
    if   style == 0: return rng.choice(WORDS)
    elif style == 1: return rng.choice(WORDS) + '_' + rng.choice(WORDS)
    elif style == 2: return rng.choice(WORDS) + '.' + rng.choice(WORDS)
    elif style == 3: return rng.choice(WORDS) + str(rng.integers(1, 99))
    else:            return rng.choice(WORDS) + rng.choice(WORDS).capitalize()


def synth_platform(platform: str, n_fake=400, n_real=400):
    """Wide-distribution synthetic. Real users in the wild range from brand-new (10 followers,
    few posts) to large (millions). Bots skew small/empty but overlap with real. Distributions
    are intentionally noisy so the model doesn't memorize any narrow rule."""
    rows = []

    # ---- Fake / bot profiles ----
    for _ in range(n_fake):
        uname = fake_username(rng)
        uname_len = max(len(uname), 4)
        digit_cnt = sum(c.isdigit() for c in uname)
        # Wider range: most bots small but a few "well-aged" bots have moderate followers
        followers = int(np.clip(rng.lognormal(mean=2.5, sigma=1.8), 0, 8000))
        following = int(np.clip(rng.lognormal(mean=5.0, sigma=1.3), 30, 8000))
        posts     = int(np.clip(rng.exponential(scale=5), 0, 80))
        has_pic   = int(rng.random() < 0.50)        # 50% — bots are getting better at this
        has_url   = int(rng.random() < 0.35)
        is_priv   = int(rng.random() < 0.18)
        bio_len   = int(np.clip(rng.exponential(scale=15), 0, 100))
        rows.append((followers, following, posts, bio_len, has_pic, has_url, is_priv,
                     uname_len, digit_cnt, digit_cnt / uname_len,
                     int(rng.integers(0, 3)), int(rng.random() < 0.3), 1))

    # ---- Genuine profiles ----
    # CRITICAL: span the full range — from brand-new accounts (5 followers, 3 posts) to
    # established (50k followers). If we only sample established users, the model thinks
    # any new user is fake.
    for _ in range(n_real):
        uname = real_username(rng, platform)
        uname_len = max(len(uname), 4)
        digit_cnt = sum(c.isdigit() for c in uname)
        # Wide log-normal centred lower — many real users have <500 followers
        followers = int(np.clip(rng.lognormal(mean=4.5, sigma=1.8), 5, 500000))
        following = int(np.clip(rng.lognormal(mean=4.8, sigma=1.0), 20, 5000))
        posts     = int(np.clip(rng.lognormal(mean=2.5, sigma=1.5), 1, 10000))
        has_pic   = int(rng.random() < 0.92)
        has_url   = int(rng.random() < 0.25)
        is_priv   = int(rng.random() < 0.30)
        bio_len   = int(np.clip(rng.normal(loc=55, scale=35), 0, 250))  # some real users have empty bios!
        rows.append((followers, following, posts, bio_len, has_pic, has_url, is_priv,
                     uname_len, digit_cnt, digit_cnt / uname_len,
                     int(rng.integers(1, 4)), int(rng.random() < 0.05), 0))

    cols = ['followers_count','following_count','posts_count','bio_length',
            'has_profile_pic','has_external_url','is_private',
            'username_length','username_digit_count','username_digit_ratio',
            'fullname_word_count','name_equals_username','is_fake']
    df = pd.DataFrame(rows, columns=cols)
    df['platform_instagram'] = int(platform == 'instagram')
    df['platform_twitter']   = int(platform == 'twitter')
    df['platform_facebook']  = int(platform == 'facebook')
    return df


print('Synthetic generator ready (400 fake + 400 real per platform = 2,400 synthetic rows total).')
print('Real Kaggle data is primary; synthetic is platform-diversity augmentation only.')
print('Distributions are WIDE — real users span new (10 followers) to huge (500k).')

## Cell 5 — Load + harmonize all sources

In [ ]:
import pandas as pd, numpy as np, glob, os

def safe_div(a, b):
    return np.where(b == 0, 0.0, a / np.maximum(b, 1))

def col(df, *candidates, default=None):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return pd.Series([default] * len(df))

# IMPORTANT: this order MUST match backend/feature_engineering.py FEATURE_COLS exactly
FEATURE_COLS = [
    'followers_count','following_count','posts_count',
    'bio_length','has_profile_pic','has_external_url','is_private',
    'username_length','username_digit_count','username_digit_ratio',
    'fullname_word_count','name_equals_username',
    'followers_following_ratio','posts_per_follower',
    'platform_instagram','platform_twitter','platform_facebook',
]

frames = []  # list of harmonized DataFrames (one per source)


def collect_csvs(root: str) -> list:
    """Recursively find all CSV files under root."""
    return glob.glob(os.path.join(root, '**', '*.csv'), recursive=True)


# ============================================================================
# IG harmonizer
# ============================================================================
def harmonize_ig(df_raw: pd.DataFrame) -> pd.DataFrame | None:
    """Map any IG-style dataset to our schema. Handles the free4ever1 column names
    and several common variants found in other Kaggle IG datasets."""
    out = pd.DataFrame()
    # Stats
    out['followers_count'] = pd.to_numeric(col(df_raw, '#followers', 'followers', 'follower_count', 'followers_count'), errors='coerce')
    out['following_count'] = pd.to_numeric(col(df_raw, '#follows',  'follows',   'following',       'following_count'), errors='coerce')
    out['posts_count']     = pd.to_numeric(col(df_raw, '#posts',    'posts',     'post_count',      'posts_count', 'media_count'), errors='coerce')
    out['bio_length']      = pd.to_numeric(col(df_raw, 'description length', 'bio_length', 'biography_length'), errors='coerce')
    out['has_profile_pic'] = pd.to_numeric(col(df_raw, 'profile pic',    'profile_pic', 'has_profile_pic', default=1), errors='coerce')
    out['has_external_url']= pd.to_numeric(col(df_raw, 'external URL',   'external_url','has_external_url', default=0), errors='coerce')
    out['is_private']      = pd.to_numeric(col(df_raw, 'private',        'is_private', default=0), errors='coerce')
    out['username_digit_ratio'] = pd.to_numeric(col(df_raw, 'nums/length username', 'username_digit_ratio'), errors='coerce')
    out['fullname_word_count']  = pd.to_numeric(col(df_raw, 'fullname words', 'fullname_word_count', default=1), errors='coerce')
    out['name_equals_username'] = pd.to_numeric(col(df_raw, 'name==username', 'name_equals_username', default=0), errors='coerce')
    out['username_length'] = 12
    out['username_digit_count'] = (out['username_digit_ratio'].fillna(0) * 12).round().astype(int)
    out['platform_instagram'], out['platform_twitter'], out['platform_facebook'] = 1, 0, 0
    out['is_fake'] = pd.to_numeric(col(df_raw, 'fake', 'is_fake', 'label', 'class'), errors='coerce')

    out = out.dropna(subset=['is_fake', 'followers_count'])
    if len(out) == 0 or out['is_fake'].nunique() < 2:
        return None
    return out


# ============================================================================
# Twitter harmonizer
# ============================================================================
def harmonize_tw(df_raw: pd.DataFrame) -> pd.DataFrame | None:
    screen_name = col(df_raw, 'screen_name', 'username', 'user_name').fillna('').astype(str)
    description = col(df_raw, 'description', 'bio').fillna('').astype(str)
    followers   = pd.to_numeric(col(df_raw, 'followers_count', 'followers'), errors='coerce').fillna(0)
    friends     = pd.to_numeric(col(df_raw, 'friends_count', 'following', 'following_count'), errors='coerce').fillna(0)
    statuses    = pd.to_numeric(col(df_raw, 'statuses_count', 'tweets_count', 'tweets'), errors='coerce').fillna(0)
    default_pic = pd.to_numeric(col(df_raw, 'default_profile_image', default=0), errors='coerce').fillna(0)
    label_raw   = col(df_raw, 'account_type', 'label', 'class', 'bot', 'is_bot').fillna('').astype(str).str.lower()

    is_fake = label_raw.apply(lambda s: 1 if any(k in s for k in ['bot','fake','spam','1','true']) else 0)
    if is_fake.sum() == 0:
        is_fake = pd.to_numeric(label_raw, errors='coerce').fillna(0).astype(int).clip(0,1)

    out = pd.DataFrame()
    out['followers_count'] = followers.astype(int)
    out['following_count'] = friends.astype(int)
    out['posts_count']     = statuses.astype(int)
    out['bio_length']      = description.str.len()
    out['has_profile_pic'] = (1 - default_pic).clip(0,1).astype(int)
    out['has_external_url']= description.str.contains(r'http', regex=True).astype(int)
    out['is_private']      = 0
    out['username_length'] = screen_name.str.len().clip(1, 30)
    out['username_digit_count'] = screen_name.str.count(r'\d')
    out['username_digit_ratio'] = safe_div(out['username_digit_count'].values, out['username_length'].values)
    out['fullname_word_count'] = 1
    out['name_equals_username'] = 0
    out['platform_instagram'], out['platform_twitter'], out['platform_facebook'] = 0, 1, 0
    out['is_fake'] = is_fake
    if out['is_fake'].nunique() < 2:
        return None
    return out


# ============================================================================
# Generic / Facebook harmonizer — used for both FB-specific datasets and
# generic "fake profile" datasets we re-label as FB.
# ============================================================================
def harmonize_generic(df_raw: pd.DataFrame, force_platform: str = 'facebook') -> pd.DataFrame | None:
    out = pd.DataFrame()
    out['followers_count'] = pd.to_numeric(col(df_raw, 'followers', 'followers_count', '#followers', 'friends'), errors='coerce').fillna(0)
    out['following_count'] = pd.to_numeric(col(df_raw, 'following', 'following_count', '#follows', 'friends_count'), errors='coerce').fillna(0)
    out['posts_count']     = pd.to_numeric(col(df_raw, 'posts', 'posts_count', '#posts', 'statuses_count', 'tweets'), errors='coerce').fillna(0)
    out['bio_length']      = pd.to_numeric(col(df_raw, 'bio_length', 'description_length', 'description length'), errors='coerce').fillna(0)
    out['has_profile_pic'] = pd.to_numeric(col(df_raw, 'profile_pic', 'has_profile_pic', 'profile pic', default=1), errors='coerce').fillna(1)
    out['has_external_url']= pd.to_numeric(col(df_raw, 'external_url', 'has_url', 'external URL', default=0), errors='coerce').fillna(0)
    out['is_private']      = pd.to_numeric(col(df_raw, 'is_private', 'private', default=0), errors='coerce').fillna(0)
    out['username_length'] = pd.to_numeric(col(df_raw, 'username_length', default=10), errors='coerce').fillna(10)
    out['username_digit_count'] = pd.to_numeric(col(df_raw, 'username_digit_count', default=0), errors='coerce').fillna(0)
    out['username_digit_ratio'] = pd.to_numeric(col(df_raw, 'username_digit_ratio', 'nums/length username'), errors='coerce')
    out['username_digit_ratio'] = out['username_digit_ratio'].fillna(safe_div(out['username_digit_count'].values, out['username_length'].values))
    out['fullname_word_count'] = pd.to_numeric(col(df_raw, 'fullname_words', 'fullname words', 'name_words', default=2), errors='coerce').fillna(2)
    out['name_equals_username'] = pd.to_numeric(col(df_raw, 'name_equals_username', 'name==username', default=0), errors='coerce').fillna(0)
    out['platform_instagram'] = int(force_platform == 'instagram')
    out['platform_twitter']   = int(force_platform == 'twitter')
    out['platform_facebook']  = int(force_platform == 'facebook')
    label = col(df_raw, 'fake', 'is_fake', 'label', 'class')
    out['is_fake'] = pd.to_numeric(label, errors='coerce').fillna(-1).astype(int).clip(-1, 1)
    out = out[out['is_fake'].isin([0, 1])]
    if len(out) == 0 or out['is_fake'].nunique() < 2:
        return None
    return out


# ============================================================================
# Load every CSV from each platform folder; try platform-specific harmonizer
# first, fall back to generic harmonizer.
# ============================================================================
def load_platform(folder: str, harmonizer, fallback_platform: str):
    loaded = []
    for path in collect_csvs(folder):
        try:
            df_raw = pd.read_csv(path, low_memory=False)
        except Exception as e:
            print(f'  skip {path}: {e}')
            continue
        h = harmonizer(df_raw)
        if h is None:
            # Try generic harmonizer as fallback
            h = harmonize_generic(df_raw, force_platform=fallback_platform)
        if h is not None:
            loaded.append(h)
            print(f'  loaded {path}: {len(h)} rows | fake_rate={h["is_fake"].mean():.2f}')
        else:
            print(f'  unusable schema: {path}  (columns: {list(df_raw.columns)[:8]})')
    if loaded:
        merged = pd.concat(loaded, ignore_index=True)
        # Dedup obvious duplicates
        merged = merged.drop_duplicates()
        return merged
    return None


print('=== Loading IG ===')
ig = load_platform('data/ig', harmonize_ig, 'instagram')
if ig is not None: frames.append(ig); print(f'TOTAL IG: {len(ig)} rows | fake_rate={ig["is_fake"].mean():.2f}')

print('\n=== Loading TW ===')
tw = load_platform('data/tw', harmonize_tw, 'twitter')
if tw is None:
    print('  No real TW data — generating synthetic')
    tw = synth_platform('twitter')
frames.append(tw)
print(f'TOTAL TW: {len(tw)} rows | fake_rate={tw["is_fake"].mean():.2f}')

print('\n=== Loading FB ===')
fb = load_platform('data/fb', lambda d: harmonize_generic(d, force_platform='facebook'), 'facebook')
if fb is None:
    print('  No real FB data — generating synthetic')
    fb = synth_platform('facebook')
frames.append(fb)
print(f'TOTAL FB: {len(fb)} rows | fake_rate={fb["is_fake"].mean():.2f}')

# If IG has very little data, add light synthetic
if ig is None or len(ig) < 200:
    print('\n  IG real data is sparse — adding synthetic IG to round out')
    frames.append(synth_platform('instagram'))


# ============================================================================
# Combine all
# ============================================================================
df = pd.concat(frames, ignore_index=True)
df['followers_following_ratio'] = safe_div(df['followers_count'].values, df['following_count'].values)
df['posts_per_follower']        = safe_div(df['posts_count'].values, df['followers_count'].values)
df = df[FEATURE_COLS + ['is_fake']].dropna()

print(f'\n=== FINAL TRAINING SET ===')
print(f'Total rows: {len(df)} | overall fake_rate={df["is_fake"].mean():.3f}')
print('\nBreakdown by platform:')
for p in ['instagram', 'twitter', 'facebook']:
    sub = df[df[f'platform_{p}'] == 1]
    print(f'  {p:10s}: {len(sub):6d} rows | fake_rate={sub["is_fake"].mean():.3f}')
df.head()

## Cell 6 — Train LightGBM

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import lightgbm as lgb
import numpy as np

X = df[FEATURE_COLS].values
y = df['is_fake'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Calibrated hyperparams — close to the original that performed well, with light
# regularization to prevent over-fitting on any single feature (esp. username digits).
# NO class_weight='balanced' because real Kaggle data is already balanced and
# rebalancing inflates fake-class confidence on real users with weak signals.
model = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=25,
    reg_alpha=0.15,
    reg_lambda=0.15,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
          callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])

pred = model.predict(X_te)
proba = model.predict_proba(X_te)[:, 1]
print(f'\n=== OVERALL ===')
print(f'AUC: {roc_auc_score(y_te, proba):.4f}')
print(classification_report(y_te, pred, digits=3))
print('Confusion matrix:')
print(confusion_matrix(y_te, pred))

# Per-platform metrics
test_df = pd.DataFrame(X_te, columns=FEATURE_COLS)
test_df['y_true'] = y_te
test_df['y_pred'] = pred
test_df['y_proba'] = proba
print(f'\n=== PER-PLATFORM AUC ===')
for p in ['instagram', 'twitter', 'facebook']:
    sub = test_df[test_df[f'platform_{p}'] == 1]
    if len(sub) > 20 and sub['y_true'].nunique() == 2:
        auc = roc_auc_score(sub['y_true'], sub['y_proba'])
        acc = (sub['y_true'] == sub['y_pred']).mean()
        print(f'  {p:10s}: n={len(sub):5d}  AUC={auc:.3f}  accuracy={acc:.3f}')

# Feature importance
imp = sorted(zip(FEATURE_COLS, model.feature_importances_), key=lambda x: -x[1])
print(f'\n=== FEATURE IMPORTANCE (lower username dominance is better) ===')
total = sum(model.feature_importances_)
for name, score in imp:
    pct = 100 * score / total if total else 0
    bar = '#' * int(pct / 2)
    print(f'  {name:30s} {score:5d}  {pct:5.1f}%  {bar}')

# ===========================================================================
# SANITY CHECKS — predict on canonical real-user and fake-bot profiles.
# If these don't pass, the model is mis-calibrated.
# ===========================================================================
def predict_one(features: dict) -> float:
    vec = np.array([[features[c] for c in FEATURE_COLS]])
    return float(model.predict_proba(vec)[0, 1])

checks = [
    # New real Twitter user with auto-suggested digit-suffix handle
    {'name': 'New real Twitter user (digit-suffix handle, 230 fol / 410 fr / 89 posts)',
     'expect': 'low',
     'feat':   {'followers_count': 230, 'following_count': 410, 'posts_count': 89,
                'bio_length': 65, 'has_profile_pic': 1, 'has_external_url': 0, 'is_private': 0,
                'username_length': 15, 'username_digit_count': 8, 'username_digit_ratio': 8/15,
                'fullname_word_count': 2, 'name_equals_username': 0,
                'followers_following_ratio': 230/410, 'posts_per_follower': 89/230,
                'platform_instagram': 0, 'platform_twitter': 1, 'platform_facebook': 0}},
    # Tiny but real new user (50 followers, posting normally)
    {'name': 'Brand-new real IG user (50 fol / 280 fr / 12 posts, has pic + bio)',
     'expect': 'low',
     'feat':   {'followers_count': 50, 'following_count': 280, 'posts_count': 12,
                'bio_length': 45, 'has_profile_pic': 1, 'has_external_url': 0, 'is_private': 1,
                'username_length': 11, 'username_digit_count': 0, 'username_digit_ratio': 0,
                'fullname_word_count': 2, 'name_equals_username': 0,
                'followers_following_ratio': 50/280, 'posts_per_follower': 12/50,
                'platform_instagram': 1, 'platform_twitter': 0, 'platform_facebook': 0}},
    # Classic bot pattern
    {'name': 'Classic IG bot (3 fol / 2800 fr / 0 posts, no pic, no bio, digit username)',
     'expect': 'high',
     'feat':   {'followers_count': 3, 'following_count': 2800, 'posts_count': 0,
                'bio_length': 0, 'has_profile_pic': 0, 'has_external_url': 0, 'is_private': 0,
                'username_length': 12, 'username_digit_count': 8, 'username_digit_ratio': 8/12,
                'fullname_word_count': 0, 'name_equals_username': 0,
                'followers_following_ratio': 3/2800, 'posts_per_follower': 0,
                'platform_instagram': 1, 'platform_twitter': 0, 'platform_facebook': 0}},
    # Established real account
    {'name': 'Established real IG account (8k fol / 600 fr / 320 posts)',
     'expect': 'low',
     'feat':   {'followers_count': 8000, 'following_count': 600, 'posts_count': 320,
                'bio_length': 120, 'has_profile_pic': 1, 'has_external_url': 1, 'is_private': 0,
                'username_length': 10, 'username_digit_count': 0, 'username_digit_ratio': 0,
                'fullname_word_count': 2, 'name_equals_username': 0,
                'followers_following_ratio': 8000/600, 'posts_per_follower': 320/8000,
                'platform_instagram': 1, 'platform_twitter': 0, 'platform_facebook': 0}},
]

print(f'\n=== SANITY CHECKS ===')
all_pass = True
for c in checks:
    p = predict_one(c['feat'])
    ok = (c['expect'] == 'low' and p < 0.5) or (c['expect'] == 'high' and p >= 0.5)
    flag = 'PASS' if ok else 'FAIL'
    if not ok: all_pass = False
    print(f'  [{flag}] expect={c["expect"]:>4s}  got={p:.3f}  | {c["name"]}')

if all_pass:
    print('\nAll sanity checks pass — model is calibrated for real-world use.')
else:
    print('\nSome checks failed — review feature importance + training distribution.')

## Cell 7 — Save and download

Drop both downloaded files into your project's `backend/` folder.

In [ ]:
import joblib, json

joblib.dump(model, 'model.pkl')
schema = {
    'feature_columns': FEATURE_COLS,
    'model_type': 'LightGBM',
    'platforms_real_data': [p for p, slug in [('instagram', ig_slug), ('twitter', tw_slug), ('facebook', fb_slug)] if slug],
    'platforms_synthetic': [p for p, slug in [('instagram', ig_slug), ('twitter', tw_slug), ('facebook', fb_slug)] if not slug],
    'auc': float(roc_auc_score(y_te, proba)),
    'n_train': int(len(X_tr)),
    'n_test': int(len(X_te)),
    'breakdown': {p: int((df[f'platform_{p}'] == 1).sum()) for p in ['instagram','twitter','facebook']},
}
with open('feature_schema.json', 'w') as f:
    json.dump(schema, f, indent=2)
print(json.dumps(schema, indent=2))

from google.colab import files
files.download('model.pkl')
files.download('feature_schema.json')
print('\nDone — both files downloaded. Drop them into your project\'s backend/ folder.')